# Two Speeds Audit Appendix

This notebook exists only to support notebook 04 and is not required reading. Each check reads the exported evidence table and prints one compact audit table.

## Check A - Is the circular matching reliable?

Publication-year agreement and lag sensitivity expose implausible circular-number matches.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
EVIDENCE_PATH = ROOT / "notebooks" / "evidence" / "04_photometry_lag.csv"
evidence = pd.read_csv(EVIDENCE_PATH)
print(f"Loaded evidence rows: {len(evidence):,}")
assert not evidence.empty, "Expected a non-empty photometry evidence CSV"
for column in ("observation_time_utc", "publication_time_utc", "created_at"):
    evidence[column] = pd.to_datetime(evidence[column], utc=True, format="mixed")
matched = evidence[evidence["circular_id"].notna()].copy()
matched["year_difference"] = (
    matched["publication_time_utc"].dt.year - matched["observation_time_utc"].dt.year
).abs()
year_groups = pd.cut(matched["year_difference"], bins=[-1, 0, 1, float("inf")], labels=["0", "1", ">1"])
rows = [{"result": "publication-year difference", "category": str(category), "n": int(count), "median_h": pd.NA, "p90_h": pd.NA} for category, count in year_groups.value_counts(sort=False).items()]
for category, values in (
    ("all matched rows", matched["lag_obs_to_publication_h"]),
    ("non-negative lag rows", matched.loc[matched["lag_obs_to_publication_h"] >= 0, "lag_obs_to_publication_h"]),
):
    rows.append({"result": "lag sensitivity", "category": category, "n": len(values), "median_h": values.median(), "p90_h": values.quantile(0.9)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

Loaded evidence rows: 7,968
                     result              category   n   median_h      p90_h
publication-year difference                     0 952       <NA>       <NA>
publication-year difference                     1   4       <NA>       <NA>
publication-year difference                    >1   2       <NA>       <NA>
            lag sensitivity      all matched rows 958  10.731605  58.269013
            lag sensitivity non-negative lag rows 948  11.047208  59.274075


Removing negative publication lags raises the median by 2.94% and the p90 by 1.73%.

## Check B - Censoring

The fixed capture date tests whether recent observations have had less time to enter SkyPortal.

In [2]:
capture_time = pd.Timestamp("2026-07-24", tz="UTC")
evidence["observable_days"] = (
    capture_time - evidence["observation_time_utc"]
).dt.total_seconds() / 86400
group_masks = [
    ("matched to a circular", evidence["circular_id"].notna()),
    ("not matched to a circular", evidence["circular_id"].isna()),
]
rows = []
for group, mask in group_masks:
    all_values = evidence.loc[mask, "lag_obs_to_skyportal_h"] / 24
    eligible = evidence.loc[mask & (evidence["observable_days"] >= 180), "lag_obs_to_skyportal_h"] / 24
    rows.append({"group": group, "all_n": len(all_values), "all_median_d": all_values.median(), "eligible_180d_n": len(eligible), "eligible_180d_median_d": eligible.median()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

                    group  all_n  all_median_d  eligible_180d_n  eligible_180d_median_d
    matched to a circular    958         4.208              575                   2.680
not matched to a circular   7010        45.528             6529                  52.288


## Check C - Bulk ingestion dates

The ten busiest database dates reveal whether rows were loaded near observation or retrospectively.

In [3]:
evidence["created_date"] = evidence["created_at"].dt.strftime("%Y-%m-%d")
evidence["age_of_data_days"] = evidence["lag_obs_to_skyportal_h"] / 24
top_dates = evidence.groupby("created_date").size().rename("n_rows").reset_index().sort_values(["n_rows", "created_date"], ascending=[False, True], kind="mergesort").head(10)
profiles = []
for row in top_dates.itertuples(index=False):
    day = evidence[evidence["created_date"] == row.created_date]
    source_counts = day.groupby("source_id").size().rename("n").reset_index().sort_values(["n", "source_id"], ascending=[False, True], kind="mergesort")
    profiles.append({
        "date": row.created_date,
        "n_rows": row.n_rows,
        "n_sources": day["source_id"].nunique(),
        "top_source": source_counts.iloc[0]["source_id"],
        "median_age_of_data_days": day["age_of_data_days"].median(),
        "max_age_of_data_days": day["age_of_data_days"].max(),
    })
top_date_values = set(top_dates["created_date"])
non_circular = evidence[evidence["circular_id"].isna() & ~evidence["created_date"].isin(top_date_values)]
result = pd.DataFrame(profiles)
result["non_circular_median_excluding_top10_d"] = (non_circular["lag_obs_to_skyportal_h"] / 24).median()
print(result.round(3).to_string(index=False))

      date  n_rows  n_sources            top_source  median_age_of_data_days  max_age_of_data_days  non_circular_median_excluding_top10_d
2024-04-26     742          1             EP240426a                  121.383               395.550                                  5.884
2024-07-16     740          2 IceCubeCascade240714A                   89.560               173.157                                  5.884
2024-06-27     642          4          ZTF23abvvlla                   99.461               395.591                                  5.884
2022-11-10     469          1             GRB221110                   84.313               370.234                                  5.884
2024-01-17     305          1             SN2023wrk                   41.075                71.550                                  5.884
2026-02-14     226          1     GCN-251222_170549                   54.025                54.121                                  5.884
2024-04-30     157          2     

## Check D - Provenance signals

The raw origin field and GCN text in `altdata` are related but not equivalent signals.

In [4]:
evidence["origin_is_gcn"] = evidence["origin_raw"].astype(str).str.strip().str.casefold().eq("gcn")
evidence["has_gcn_reference"] = evidence["has_gcn_reference"].astype("boolean").fillna(False)
rows = []
for origin_value in (False, True):
    for reference_value in (False, True):
        mask = (evidence["origin_is_gcn"] == origin_value) & (evidence["has_gcn_reference"] == reference_value)
        rows.append({"result": "signal cross-tab", "definition": pd.NA, "group": pd.NA, "origin_is_gcn": origin_value, "has_gcn_reference": reference_value, "n": int(mask.sum()), "median_d": pd.NA})
for definition, signal in (
    ("origin field", evidence["origin_is_gcn"]),
    ("altdata GCN reference", evidence["has_gcn_reference"]),
):
    for group, mask in (("GCN-signaled", signal), ("other", ~signal)):
        values = evidence.loc[mask, "lag_obs_to_skyportal_h"] / 24
        rows.append({"result": "group median", "definition": definition, "group": group, "origin_is_gcn": pd.NA, "has_gcn_reference": pd.NA, "n": len(values), "median_d": values.median()})
print(pd.DataFrame(rows).round(3).to_string(index=False))

          result            definition        group origin_is_gcn has_gcn_reference    n   median_d
signal cross-tab                  <NA>         <NA>         False             False 6699       <NA>
signal cross-tab                  <NA>         <NA>         False              True  272       <NA>
signal cross-tab                  <NA>         <NA>          True             False   30       <NA>
signal cross-tab                  <NA>         <NA>          True              True  967       <NA>
    group median          origin field GCN-signaled          <NA>              <NA>  997   4.350768
    group median          origin field        other          <NA>              <NA> 6971  46.049869
    group median altdata GCN reference GCN-signaled          <NA>              <NA> 1239   4.429629
    group median altdata GCN reference        other          <NA>              <NA> 6729   44.51505
